In [ ]:
import pyodbc, struct, json as _json, traceback as _tb

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"

def write_diag(name, obj):
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}/Files/diagnostics/{name}.json"
    notebookutils.fs.put(path, _json.dumps(obj, indent=2, default=str), overwrite=True)
    print(f"wrote {path}")

try:
    server = "4oa7g2g2i24uposxn4zcxdynue-y2u3efqymrpurbs7yr2efofwyy.datawarehouse.fabric.microsoft.com"
    database = "wh_gold"
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server};DATABASE={database};Encrypt=yes;"
    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, autocommit=True)
    cursor = conn.cursor()
    print("connected")
except Exception as e:
    write_diag("validate_sm_connect", {"status": "failed", "error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
try:
    def kpi_for(where_clause):
        cursor.execute(f"""
            SELECT
                COUNT(*) AS flights,
                SUM(CASE WHEN is_cancelled = 0 AND dep_delay_min <= 0 THEN 1 ELSE 0 END) * 1.0
                    / SUM(CASE WHEN is_cancelled = 0 THEN 1 ELSE 0 END) AS d0,
                SUM(CASE WHEN is_cancelled = 0 AND arr_delay_min <= 14 THEN 1 ELSE 0 END) * 1.0
                    / SUM(CASE WHEN is_cancelled = 0 THEN 1 ELSE 0 END) AS a14,
                1.0 - SUM(CASE WHEN is_cancelled = 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS cf
            FROM dbo.fact_flight_leg
            WHERE {where_clause}
        """)
        r = cursor.fetchone()
        return {"flights": r[0], "d0": float(r[1]), "a14": float(r[2]), "cf": float(r[3])}

    slices = {
        "overall": kpi_for("1=1"),
        "dfw_aug_2026": kpi_for("origin_airport_key = 'DFW' AND sched_dep_utc >= '2026-08-01' AND sched_dep_utc < '2026-09-01'"),
        "aa_2024": kpi_for("marketing_carrier_key = 'AA' AND sched_dep_utc >= '2024-01-01' AND sched_dep_utc < '2025-01-01'"),
    }
    print(_json.dumps(slices, indent=2))
    write_diag("validate_sm_slices", slices)
except Exception as e:
    write_diag("validate_sm_slices_ERROR", {"error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
try:
    cursor.execute("SELECT COUNT(*) FROM sys.tables WHERE name = 'fact_station_day_scorecard'")
    exists = cursor.fetchone()[0]
    result = {"table_exists": bool(exists)}
    if exists:
        cursor.execute("SELECT COUNT(*) FROM dbo.fact_station_day_scorecard")
        result["row_count"] = cursor.fetchone()[0]
    print(_json.dumps(result, indent=2))
    write_diag("validate_sm_scorecard_check", result)
except Exception as e:
    write_diag("validate_sm_scorecard_check_ERROR", {"error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
try:
    result = {}
    for t in ["fact_tsa_departure_hour_profile", "fact_cbp_hour_profile"]:
        cursor.execute("SELECT COUNT(*) FROM sys.tables WHERE name = ?", t)
        exists = cursor.fetchone()[0]
        row = {"table_exists": bool(exists)}
        if exists:
            cursor.execute(f"SELECT COUNT(*) FROM dbo.{t}")
            row["row_count"] = cursor.fetchone()[0]
        result[t] = row
    cursor.execute("SELECT name, type_desc, create_date FROM sys.tables WHERE name LIKE '%passenger%' OR name LIKE '%tsa%' OR name LIKE '%cbp%'")
    result["matching_tables"] = [{"name": r[0], "type": r[1], "created": str(r[2])} for r in cursor.fetchall()]
    print(_json.dumps(result, indent=2))
    write_diag("validate_sm_passenger_flow_check", result)
except Exception as e:
    write_diag("validate_sm_passenger_flow_check_ERROR", {"error": repr(e), "traceback": _tb.format_exc()})
    raise
